# Tiny Stories GPT - Story Generation

Load a pretrained GPT model checkpoint and generate creative stories.

## 1. Import Required Libraries

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
from pathlib import Path
import json
import time

## 2. Define Model Architecture

In [3]:
class Head(nn.Module):
    """Single head of self-attention"""
    
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel"""
    
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(head_size * n_head, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """Simple feed-forward layer"""
    
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block"""
    
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    """GPT Language Model"""
    
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout, device):
        super().__init__()
        self.block_size = block_size
        self.device = device
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=self.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Generate new tokens"""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


print("✓ Model architecture defined")

✓ Model architecture defined


## 3. Load the Pretrained Model

In [5]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Path to checkpoint
checkpoint_path = Path('./gpt_5_3500.pth')

# Model hyperparameters (must match the trained model)
vocab_size = 50257  # GPT-2 tokenizer vocab size
block_size = 64
n_embd = 512
n_head = 12
n_layer = 12
dropout = 0.2

print(f"\nModel Configuration:")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Block size: {block_size}")
print(f"  Embedding dimension: {n_embd}")
print(f"  Attention heads: {n_head}")
print(f"  Transformer layers: {n_layer}")
print(f"  Dropout: {dropout}")

# Initialize model
print(f"\nLoading model from: {checkpoint_path}")
model = GPTLanguageModel(vocab_size, block_size, n_embd, n_head, n_layer, dropout, device)
model = model.to(device)

# Load checkpoint
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# Count parameters
num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {num_params:.2f}M")
print(f"✓ Model loaded successfully")

# Initialize tokenizer
enc = tiktoken.get_encoding("gpt2")
print(f"✓ Tokenizer initialized")

Using device: cuda

Model Configuration:
  Vocabulary size: 50257
  Block size: 64
  Embedding dimension: 512
  Attention heads: 12
  Transformer layers: 12
  Dropout: 0.2

Loading model from: gpt_5_3500.pth


Model parameters: 89.16M
✓ Model loaded successfully
✓ Tokenizer initialized


## 4. Define Text Generation Parameters

In [6]:
# Generation parameters
GENERATION_CONFIG = {
    'max_tokens': 500,          # Maximum new tokens to generate
    'num_stories': 3,           # Number of stories to generate
}

# Different prompts for story generation
PROMPTS = [
    "Once upon a time",
    "There was a little",
    "In a magical forest",
    "One sunny day",
    "A long time ago",
]

print("Generation Configuration:")
for key, value in GENERATION_CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nAvailable prompts: {len(PROMPTS)}")

Generation Configuration:
  max_tokens: 500
  num_stories: 3

Available prompts: 5


## 5. Generate Stories

In [7]:
def generate_story(prompt, max_tokens=500):
    """Generate a single story from a prompt"""
    # Encode prompt
    encoded_prompt = enc.encode(prompt)
    context = torch.tensor(encoded_prompt, dtype=torch.long, device=device).unsqueeze(0)
    
    # Generate
    start_time = time.time()
    with torch.no_grad():
        generated = model.generate(context, max_new_tokens=max_tokens)
    elapsed_time = time.time() - start_time
    
    # Decode
    generated_text = enc.decode(generated[0].tolist())
    
    return generated_text, elapsed_time


# Generate multiple stories
generated_stories = []
max_tokens = GENERATION_CONFIG['max_tokens']
num_stories = GENERATION_CONFIG['num_stories']

for i in range(num_stories):
    prompt = PROMPTS[i % len(PROMPTS)]
    print(f"\n{'='*70}")
    print(f"Story {i+1}/{num_stories} | Prompt: {prompt!r}")
    print(f"{'='*70}")
    
    story, elapsed = generate_story(prompt, max_tokens)
    generated_stories.append({'prompt': prompt, 'story': story, 'time': elapsed})
    
    print(story)
    print(f"\n⏱️  Generation time: {elapsed:.2f}s")

print(f"\n{'='*70}")
print(f"✓ Generated {len(generated_stories)} stories")
print(f"{'='*70}")


Story 1/3 | Prompt: 'Once upon a time'
Once upon a time, there was a clown. The clown had bright yellow skin and a friendly smile. He loved to sit in the same spot on a sunny day. Whenever he felt sad, he liked to sit there and watch the other kids having fun.

One day, he took a big red ball out of his hat and wanted to bounce around. So he bounced to the sidelines and wished that he could fly like them. He bounced around and in the morning, until he was too tired to know how much fun it would be to bounce around.

He hopped about it, bouncing even higher and higher with all his might. After a few minutes, he stopped bouncing and smiled. He felt like he had play and had lots of fun.

The persistent bunny hopped around the garden, bouncing around and having so much fun. He thought the rock was very clever and kept bouncing for longer. Eventually, he hopped just like he was having fun!
Once upon a time there were two best friends, Joe and Bob. They were very good boys and liked to shar

## 6. Display and Analyze Generated Stories

In [8]:
# Analyze generated stories
print("\n" + "="*70)
print("GENERATION STATISTICS")
print("="*70)

total_time = sum(s['time'] for s in generated_stories)
avg_time = total_time / len(generated_stories)

print(f"\nTotal stories generated: {len(generated_stories)}")
print(f"Total time: {total_time:.2f}s")
print(f"Average time per story: {avg_time:.2f}s")

print(f"\nPer-story statistics:")
for i, story_data in enumerate(generated_stories):
    prompt = story_data['prompt']
    text = story_data['story']
    elapsed = story_data['time']
    
    num_tokens = len(enc.encode(text))
    tokens_per_sec = num_tokens / elapsed if elapsed > 0 else 0
    
    print(f"\n  Story {i+1}:")
    print(f"    Prompt: {prompt!r}")
    print(f"    Tokens: {num_tokens}")
    print(f"    Time: {elapsed:.2f}s")
    print(f"    Speed: {tokens_per_sec:.1f} tokens/sec")


GENERATION STATISTICS

Total stories generated: 3
Total time: 62.52s
Average time per story: 20.84s

Per-story statistics:

  Story 1:
    Prompt: 'Once upon a time'
    Tokens: 504
    Time: 21.32s
    Speed: 23.6 tokens/sec

  Story 2:
    Prompt: 'There was a little'
    Tokens: 504
    Time: 20.84s
    Speed: 24.2 tokens/sec

  Story 3:
    Prompt: 'In a magical forest'
    Tokens: 504
    Time: 20.37s
    Speed: 24.7 tokens/sec


## 7. Generate Custom Story

In [ ]:
# Generate a story with your own prompt
custom_prompt = "A clever rabbit"  # Change this to your desired prompt
print(f"Generating story with custom prompt: {custom_prompt!r}\n")

custom_story, custom_time = generate_story(custom_prompt, max_tokens=800)
print(custom_story)
print(f"\n⏱️  Generation time: {custom_time:.2f}s")

## 8. Save Generated Stories

In [ ]:
# Save all generated stories to a text file
output_file = Path('./generated_stories.txt')

with open(output_file, 'w') as f:
    for i, story_data in enumerate(generated_stories):
        f.write(f"{'='*70}\n")
        f.write(f"Story {i+1}\n")
        f.write(f"Prompt: {story_data['prompt']}\n")
        f.write(f"Generation time: {story_data['time']:.2f}s\n")
        f.write(f"{'='*70}\n\n")
        f.write(story_data['story'])
        f.write(f"\n\n")

print(f"✓ Stories saved to: {output_file}")

## 9. Fine-tune on SimpleStories by Story Type

This section fine-tunes the loaded checkpoint on user-specified story types from the SimpleStories dataset.

Notes:
- SimpleStories is very large, so this uses streaming and a capped sample budget.
- The dataset may not provide an explicit `story_type` field; we infer type using keyword rules.
- You can edit `TARGET_STORY_TYPES` below to pick which style(s) to train on.

In [10]:
from datasets import load_dataset
from itertools import islice
import random

# Pick one or more target styles to fine-tune on.
TARGET_STORY_TYPES = ["animal", "friendship"]

# Keyword-based type definitions used to infer type from text.
STORY_TYPE_KEYWORDS = {
    "animal": ["cat", "dog", "rabbit", "bird", "lion", "bear", "fox", "animal"],
    "friendship": ["friend", "friends", "kind", "share", "help", "together"],
    "bedtime": ["sleep", "night", "dream", "bed", "moon", "stars"],
    "school": ["school", "teacher", "class", "learn", "homework", "student"],
    "magic": ["magic", "wizard", "fairy", "spell", "enchanted", "dragon"],
    "adventure": ["journey", "adventure", "explore", "map", "treasure", "quest"],
}

# Fine-tuning budget (keep modest for notebook runs).
MAX_CANDIDATES_TO_SCAN = 80000
MAX_TRAIN_STORIES = 25000
MAX_VAL_STORIES = 1200
FINETUNE_EPOCHS = 1
FINETUNE_BATCH_SIZE = 8
FINETUNE_GRAD_ACCUM = 4
FINETUNE_LR = 5e-5
FINETUNE_WARMUP_STEPS = 100
EVAL_EVERY_STEPS = 200

# Reproducibility
random.seed(42)
torch.manual_seed(42)

print("Selected target types:", TARGET_STORY_TYPES)
print("Type dictionary keys:", sorted(STORY_TYPE_KEYWORDS.keys()))

Selected target types: ['animal', 'friendship']
Type dictionary keys: ['adventure', 'animal', 'bedtime', 'friendship', 'magic', 'school']


In [25]:
# Choose stories by dataset "style" field
# Edit this list to the style(s) you want.
TARGET_STYLES = ["tragic"]
MAX_STYLE_SCAN = 100000
MAX_STYLE_MATCHES = 5000

style_ds = load_dataset("SimpleStories/SimpleStories", split="train", streaming=True)

style_selected_stories = []
style_counts = {}
seen = 0

for ex in islice(style_ds, MAX_STYLE_SCAN):
    seen += 1
    style = ex.get("style", None)
    story_text = ex.get("story", None)

    if not isinstance(style, str) or not isinstance(story_text, str):
        continue

    style = style.strip().lower()
    style_counts[style] = style_counts.get(style, 0) + 1

    if style in [s.lower() for s in TARGET_STYLES]:
        style_selected_stories.append(
            {
                "style": style,
                "story": story_text,
                "topic": ex.get("topic", None),
                "theme": ex.get("theme", None),
            }
        )
        if len(style_selected_stories) >= MAX_STYLE_MATCHES:
            break

print(f"Scanned: {seen:,}")
print(f"Matched ({TARGET_STYLES}): {len(style_selected_stories):,}")

print("\nTop observed styles in scan:")
for s, c in sorted(style_counts.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"  {s}: {c}")

if style_selected_stories:
    print("\nPreview:")
    for j, row in enumerate(style_selected_stories[:3], 1):
        preview = row["story"][:220].replace("\n", " ")
        print(f"\n{j}) style={row['style']!r}, topic={row['topic']!r}, theme={row['theme']!r}")
        print(f"   {preview}...")

Scanned: 100,000
Matched (['tragic']): 4,252

Top observed styles in scan:
  minimalist: 4552
  whimsical: 4435
  classic: 4429
  playful: 4423
  action-packed: 4419
  surreal: 4404
  lighthearted: 4399
  mystical: 4378
  mythological: 4373
  fairy tale-like: 4372

Preview:

1) style='tragic', topic='living objects', theme='Logic'
   Brave waves crashed against the rocks, but a tiny fish named Alex did not care. He lived in a coral reef, surrounded by colors and life. Each day, Alex swam, dodging bigger fish and trying to find food. Yet, he always fe...

2) style='tragic', topic='robots and technology', theme='Resourcefulness'
   Hopeful music filled the air as Alex played a game with his robot, Buddy. They laughed and danced in the light of the screen. One day, the game crashed, and Buddy fell still. Alex felt a sharp pain in his chest. "No, Bud...

3) style='tragic', topic='the sky', theme='Discovery'
   By the river, a girl will skip stones. Each stone will make a small splash, echo

In [26]:
# Build fine-tune split strictly from your selected styles (TARGET_STYLES)
# Requires the previous style-selection cell to be run first.
if "style_selected_stories" not in globals() or len(style_selected_stories) == 0:
    raise ValueError(
        "No style-selected stories found. Run the style selection cell above and/or change TARGET_STYLES."
    )

# Extract text only from the style-filtered rows
all_style_texts = [row["story"] for row in style_selected_stories if isinstance(row.get("story"), str) and row["story"].strip()]

if len(all_style_texts) == 0:
    raise ValueError("Selected styles produced no valid story text.")

# Shuffle once for a clean train/val split
random.seed(42)
random.shuffle(all_style_texts)

# 95/5 split, capped by your fine-tune limits
split_idx = max(1, int(0.95 * len(all_style_texts)))
train_pool = all_style_texts[:split_idx]
val_pool = all_style_texts[split_idx:]

train_texts = train_pool[:MAX_TRAIN_STORIES]
val_texts = val_pool[:MAX_VAL_STORIES]

# Guarantee non-empty validation set
if len(val_texts) == 0:
    fallback_n = min(max(1, len(train_texts) // 20), len(train_texts))
    val_texts = train_texts[:fallback_n]

style_summary = {
    "target_styles": TARGET_STYLES,
    "matched_total": len(all_style_texts),
    "train_count": len(train_texts),
    "val_count": len(val_texts),
}

print("Style-only fine-tuning summary:")
for k, v in style_summary.items():
    print(f"  {k}: {v}")

Style-only fine-tuning summary:
  target_styles: ['tragic']
  matched_total: 4252
  train_count: 4039
  val_count: 213


In [27]:
# Build token tensor for continuation training
finetune_text = "\n".join(train_texts)
val_text = "\n".join(val_texts) if len(val_texts) > 0 else finetune_text[: max(1000, len(finetune_text)//20)]

finetune_train_data = torch.tensor(enc.encode(finetune_text), dtype=torch.long)
finetune_val_data = torch.tensor(enc.encode(val_text), dtype=torch.long)

print("Tokenized fine-tuning corpus:")
print(f"  train tokens: {len(finetune_train_data):,}")
print(f"  val tokens:   {len(finetune_val_data):,}")


def get_finetune_batch(split, batch_size=FINETUNE_BATCH_SIZE):
    data = finetune_train_data if split == "train" else finetune_val_data
    if len(data) <= block_size + 1:
        raise ValueError("Fine-tune data is too short for current block_size.")
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_finetune_loss(eval_iters=50):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_finetune_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

Tokenized fine-tuning corpus:
  train tokens: 1,226,214
  val tokens:   61,007


In [28]:
# Optional: snapshot baseline model for before/after comparison
baseline_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
print("Saved baseline snapshot in memory.")

# Fine-tuning optimizer
finetune_optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR)

# Estimate run length by token budget
steps_per_epoch = max(1, len(finetune_train_data) // (FINETUNE_BATCH_SIZE * block_size))
total_steps = max(1, FINETUNE_EPOCHS * steps_per_epoch)

print("Fine-tuning setup:")
print(f"  epochs: {FINETUNE_EPOCHS}")
print(f"  steps_per_epoch: {steps_per_epoch}")
print(f"  total_steps: {total_steps}")
print(f"  batch_size: {FINETUNE_BATCH_SIZE}")
print(f"  grad_accum: {FINETUNE_GRAD_ACCUM}")
print(f"  lr: {FINETUNE_LR}")

ft_history = {"step": [], "train_loss": [], "val_loss": []}
model.train()

global_step = 0
for epoch in range(FINETUNE_EPOCHS):
    for _ in range(steps_per_epoch):
        global_step += 1

        # Linear warmup
        if global_step <= FINETUNE_WARMUP_STEPS:
            warmup_scale = global_step / max(1, FINETUNE_WARMUP_STEPS)
            for pg in finetune_optimizer.param_groups:
                pg["lr"] = FINETUNE_LR * warmup_scale

        finetune_optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        for _acc in range(FINETUNE_GRAD_ACCUM):
            xb, yb = get_finetune_batch("train")
            _, loss = model(xb, yb)
            (loss / FINETUNE_GRAD_ACCUM).backward()
            running_loss += loss.item()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        finetune_optimizer.step()

        if global_step % EVAL_EVERY_STEPS == 0 or global_step == 1 or global_step == total_steps:
            eval_metrics = estimate_finetune_loss(eval_iters=30)
            mean_train_loss = running_loss / FINETUNE_GRAD_ACCUM
            ft_history["step"].append(global_step)
            ft_history["train_loss"].append(mean_train_loss)
            ft_history["val_loss"].append(eval_metrics["val"])
            print(
                f"step {global_step:5d}/{total_steps} | "
                f"train_loss {mean_train_loss:.4f} | val_loss {eval_metrics['val']:.4f}"
            )

print("Fine-tuning complete.")

Saved baseline snapshot in memory.
Fine-tuning setup:
  epochs: 1
  steps_per_epoch: 2394
  total_steps: 2394
  batch_size: 8
  grad_accum: 4
  lr: 5e-05
step     1/2394 | train_loss 3.4588 | val_loss 3.2594
step   200/2394 | train_loss 2.9206 | val_loss 3.0729
step   400/2394 | train_loss 3.0245 | val_loss 2.9054
step   600/2394 | train_loss 2.8842 | val_loss 2.8487
step   800/2394 | train_loss 2.7481 | val_loss 2.7644
step  1000/2394 | train_loss 2.7477 | val_loss 2.6875
step  1200/2394 | train_loss 2.6658 | val_loss 2.7529
step  1400/2394 | train_loss 2.6028 | val_loss 2.6595
step  1600/2394 | train_loss 2.6131 | val_loss 2.6621
step  1800/2394 | train_loss 2.6303 | val_loss 2.6568
step  2000/2394 | train_loss 2.7266 | val_loss 2.6499
step  2200/2394 | train_loss 2.4686 | val_loss 2.6470
step  2394/2394 | train_loss 2.4967 | val_loss 2.5882
Fine-tuning complete.


In [29]:
# Save fine-tuned checkpoint
finetuned_ckpt = Path("./gpt_5_3500_simplestories_finetuned.pth")
torch.save(model.state_dict(), finetuned_ckpt)
print(f"Saved fine-tuned checkpoint to: {finetuned_ckpt}")

# Save training curve data
ft_history_path = Path("./finetune_history_simplestories.json")
with open(ft_history_path, "w") as f:
    json.dump(ft_history, f, indent=2)
print(f"Saved fine-tune history to: {ft_history_path}")

Saved fine-tuned checkpoint to: gpt_5_3500_simplestories_finetuned.pth
Saved fine-tune history to: finetune_history_simplestories.json


In [30]:
# Compare generation after fine-tuning
comparison_prompts = [
    "A little fox and her friend",
    "At bedtime, the child",
    "In school today,",
]

for p in comparison_prompts:
    print("\n" + "=" * 80)
    print(f"Prompt: {p!r}")
    print("=" * 80)
    generated, elapsed = generate_story(p, max_tokens=250)
    print(generated)
    print(f"\nGeneration time: {elapsed:.2f}s")


Prompt: 'A little fox and her friend'


A little fox and her friend stood on a log, looking for signs of sadness. Each day, their mother used to show kindness and shared secrets among the animals. They shared a tale of a fox who once felt the ache of strength. 

As night approached fear, the boy left the bear entered, desperate to embrace one another. He wanted to fix the fox's pain without trust. But the door closed shut behind him! The boy called, but the sound grew louder now watches in fear. The bear stayed firm, thinking that the boy had given. It was a era, hiding a family another boy no longer lost.
One day, a boy named Leo found a small screen in his attic. It was a painting of a unicorn named Leo. He was searching for a way but had only many toys. Leo remembered a time when he looked up at his flowers, thinking of their happy faces. He had an idea. 

With one rainy hand, Leo filled his hands with seeds and water. "I can help my new colors!" he shouted gleaming. As he began to sing, he felt a strange light up. He kne